# EDA & Baseline Model — German Credit + Synthetic Alt-Data

This notebook performs exploratory data analysis on the UCI German Credit Dataset augmented with synthetic alternative-data features, and trains a baseline logistic regression model.

**Data Governance Notice:** All alternative-data features are synthetically generated for demonstration purposes. No real telco, e-wallet, or personal data was used or accessed.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_pipeline import load_german_credit_data, load_and_prepare
from src.synthetic_generator import SyntheticAltDataGenerator

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Raw Data

In [ ]:
raw_df = load_german_credit_data()
print(f"Raw data: {raw_df.shape[0]} samples, {raw_df.shape[1]} columns")
print(f"\nDefault distribution:")
print(raw_df['default'].value_counts())
print(f"\nDefault rate: {raw_df['default'].mean():.3f}")

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

raw_df['default'].value_counts().plot(kind='barh', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Default Distribution')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Class')
axes[0].set_xticklabels(['Good Credit (0)', 'Default (1)'])

raw_df['default'].value_counts(normalize=True).plot(kind='barh', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Default Distribution (%)')
axes[1].set_xlabel('Proportion')
axes[1].set_ylabel('Class')
axes[1].set_xticklabels(['Good Credit (0)', 'Default (1)'])

plt.tight_layout()
plt.show()

## 2. Load Full Pipeline (Base + Synthetic Features)

In [ ]:
data = load_and_prepare(include_synthetic=True, seed=42)

print(f"Train: {data['n_train']} samples, {data['n_features']} features")
print(f"Test:  {data['n_test']} samples")
print(f"Default rate — Train: {data['default_rate_train']:.3f}, Test: {data['default_rate_test']:.3f}")
print(f"\nTotal features: {data['n_features']}")
print(f"Synthetic features: {sum(1 for f in data['feature_names'] if 'synthetic_' in f)}")
print(f"Base features: {sum(1 for f in data['feature_names'] if 'synthetic_' not in f)}")

In [ ]:
# Feature documentation
gen = SyntheticAltDataGenerator(seed=42)
doc_df = gen.generate_documentation_df()
doc_df

## 3. Feature Correlation Analysis

In [ ]:
# Correlation of synthetic features with default (using training data labels)
synthetic_names = [f for f in data['feature_names'] if 'synthetic_' in f and not any(s in f for s in ['income_bracket', 'region', 'age_band'])]
synthetic_indices = [data['feature_names'].index(f) for f in synthetic_names]

correlations = []
for i, name in zip(synthetic_indices, synthetic_names):
    corr = np.corrcoef(data['X_train'][:, i], data['y_train'])[0, 1]
    correlations.append({'feature': name, 'correlation_with_default': corr})

corr_df = pd.DataFrame(correlations).sort_values('correlation_with_default', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['red' if v > 0 else 'green' for v in corr_df['correlation_with_default']]
ax.barh(corr_df['feature'], corr_df['correlation_with_default'], color=colors)
ax.set_xlabel('Correlation with Default (positive = higher risk)')
ax.set_title('Synthetic Feature — Default Correlation')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 4. Baseline Model Results

In [ ]:
from src.train_baseline import train_baseline

result = train_baseline(
    data['X_train'], data['X_test'],
    data['y_train'], data['y_test'],
    data['feature_names'],
)

print(f"ROC-AUC: {result['metrics']['roc_auc']:.4f}")
print(f"Accuracy: {result['metrics']['accuracy']:.4f}")

## 5. SHAP Explainability Demo

In [ ]:
from src.explainability import SHAPExplainer
import joblib

# Load the challenger model
model = joblib.load(PROJECT_ROOT / 'models' / 'challenger_xgboost.pkl')
feature_names = data['feature_names']

# Create explainer
explainer = SHAPExplainer(model, feature_names, model_type='tree')
explainer.fit(data['X_train'], sample_size=50)

# Explain a single prediction
sample_idx = 0
explanation = explainer.explain_single(data['X_test'][sample_idx:sample_idx+1], return_dict=True)

print(f"Sample {sample_idx}:")
print(f"  Actual label: {data['y_test'][sample_idx]}")
print(f"  Base value: {explanation['base_value']:.4f}")
print(f"  SHAP prediction: {explanation['prediction_shap']:.4f}")
print(f"\nTop 10 attributions:")
for attr in explanation['attributions'][:10]:
    direction = '↑ risk' if attr['shap_value'] > 0 else '↓ risk'
    print(f"  {attr['feature']}: {attr['shap_value']:+.4f} ({direction})")

In [ ]:
# SHAP summary visualization (top 15 features)
shap_values = explainer.explain_batch(data['X_test'][:200])
feature_importance = []
for i, name in enumerate(feature_names):
    feature_importance.append({
        'feature': name,
        'mean_abs_shap': float(np.mean(np.abs(shap_values[:, i])))
    })

fi_df = pd.DataFrame(feature_importance).sort_values('mean_abs_shap', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df['feature'][::-1], fi_df['mean_abs_shap'][::-1], color='#3498db')
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Top 15 Feature Importance (SHAP)')
plt.tight_layout()
plt.show()

## 6. STARS Framework Compliance Summary

| Pillar | Status |
|--------|--------|
| **S**ustainability | ✅ Borderline scores flagged for human review |
| **T**ransparency | ✅ SHAP explanations per prediction |
| **A**ccountability | ✅ Auto-generated Model Card |
| **R**esponsibility | ✅ Fairness audit with CI gate |
| **S**ecurity | ✅ No real PII, synthetic-data policy enforced |

---

**⚠️ Disclaimer:** This notebook is for demonstration purposes only. All alternative-data features are synthetic. This model is NOT intended for production credit decisions.